# 03 — Evaluate base / SFT / GPTQ with lm-evaluation-harness

Run this after notebooks 01 and 02 have produced `sft_cfg['merged_dir']` and `gptq_cfg['output_dir']` (either in this Colab session, or reloaded from the Hub if you pushed them).

Each cell below is independent and can be re-run on its own — each is one Colab session's worth of compute across 6 benchmark tasks on an 8B-parameter multimodal model, so budget accordingly against the free-tier usage limits. Add a numeric 4th arg to `run_eval.sh` to `--limit` examples-per-task for a fast smoke test first.

## (If starting a fresh session) reinstall + reclone

In [ ]:
# !git clone https://github.com/Dushyantgehlot/slm_prod.git
# %cd slm_prod
# !pip install -q -r requirements-colab.txt && pip install -q -e .


## Smoke test (fast, low `--limit`) — confirms the harness + model args are wired correctly

In [ ]:
!bash eval/run_eval.sh base "pretrained=google/gemma-4-E4B,dtype=bfloat16" 20


## Full eval: base model

In [ ]:
!bash eval/run_eval.sh base "pretrained=google/gemma-4-E4B,dtype=bfloat16"


## Full eval: SFT model (merged)

In [ ]:
from slm_prod.utils import load_config
sft_cfg = load_config("sft.yaml")
!bash eval/run_eval.sh sft "pretrained={sft_cfg['merged_dir']},dtype=bfloat16"


## Full eval: GPTQ 4-bit model

In [ ]:
gptq_cfg = load_config("gptq.yaml")
!bash eval/run_eval.sh gptq "pretrained={gptq_cfg['output_dir']}"


## Quick peek at results

In [ ]:
import json
from pathlib import Path

for label in ["base", "sft", "gptq"]:
    result_files = list(Path(f"eval/results/{label}").rglob("results*.json"))
    if not result_files:
        print(label, "— no results yet")
        continue
    with open(result_files[0]) as f:
        data = json.load(f)
    print(label, "->", {k: v.get("acc,none", v) for k, v in data["results"].items()})


Continue with **04_results_analysis.ipynb** to build the final comparison table and plots.